# Machine Learning From Scratch

In [ ]:
# Add import statements here
from sklearn.tree import DecisionTreeClassifier
from scipy.stats import mode
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import tensorflow as tf

In [ ]:
# To access files in your Google Drive, run this block and follow the instructions
from google.colab import drive
drive.mount('/content/gdrive')

In [ ]:
# To test if the above block worked, run this block
!ls '/content/gdrive/My Drive/HW5_Data_Files'

## Neural Network

The `neural_network` function creates a model that learns to classify handwritten digits.

Inputs:
* `X_train` is the training data
* `y_train` are the training labels
* `X_test` is the testing data
* `y_test` are the testing labels

Outputs: 
* `test_loss` is the loss after evaluating the testing dataset
* `test_acc` is the accuracy after evaluating the testing dataset
* `predictions` are the models predictions of the testing dataset

Note: Have fun and be creative with this implementation!


In [ ]:
def neural_network(x_train, y_train, x_test, y_test):

  # Implement model
  model = tf.keras.Sequential()
  model.add(tf.keras.Input(shape=(28,28)))
  model.add(tf.keras.layers.Flatten())
  model.add(tf.keras.layers.Dense(16, activation="tanh"))
  model.add(tf.keras.layers.Dense(16, activation="tanh"))
  model.add(tf.keras.layers.Dropout(0.2))
  model.add(tf.keras.layers.Dense(16, activation="tanh"))
  model.add(tf.keras.layers.Dense(10))

  # Feel free to change this up, but leave it at first
  model.compile(optimizer='adam',
                loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                metrics=['accuracy'])
  
  # Fit and evaluate
  history = model.fit(x_train, y_train, batch_size=100, epochs=10)
  results = model.evaluate(x_test, y_test, batch_size=100)
  test_loss = results[0]
  test_acc = results[1]

  # Calculate predictions
  predictions = model.predict(x_test)

  return test_loss, test_acc, predictions

## Run and Plot

Run your neural network code and plot figures below

In [ ]:
# Other neural network code here:

# Load data
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()


In [ ]:
fig = plt.figure(figsize=(10, 10))
fig.subplots_adjust(hspace=0.2, wspace=0.2)
for i in range(0, 25):
  fig.add_subplot(5, 5, i+1)
  plt.imshow(x_train[i, :])
plt.show()

In [ ]:
x_train_scaled = x_train.astype(float)
x_test_scaled = x_test.astype(float)
for i in range(0, x_train.shape[0]):
  x_train_scaled[i, :] /= 255
for i in range(0, x_test.shape[0]):
  x_test_scaled[i, :] /= 255

In [ ]:
test_loss, test_acc, predictions = neural_network(x_train_scaled, y_train, x_test_scaled, y_test)
print(test_loss, " | ", test_acc)

In [ ]:
numbers = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

fig = plt.figure(figsize=(30, 30))
fig.subplots_adjust(hspace=1, wspace=0)
num_points = 20
for i in range(0, num_points):
  fig.add_subplot(num_points, 2, 2*i + 1)
  plt.xlabel("Number")
  plt.ylabel("Prediction")
  plt.bar(numbers, predictions[i])
  fig.add_subplot(num_points, 2, 2*(i+1))
  plt.imshow(x_test[i, :])
plt.show()

## Random Forest

The `random_forest` function learns an ensemble of numBags CART decision trees using a random subset of the features at each split on the input dataset and also plots the  out-of-bag error as a function of the number of bags

Inputs:
* `X_train` is the training data
* `y_train` are the training labels
* `X_test` is the testing data
* `y_test` are the testing labels
* `num_bags` is the number of trees to learn in the ensemble
* `m` is the number of randomly selected features to consider at each split

Outputs: 
* `out_of_bag_error` is the out-of-bag classification error of the final learned ensemble
* `test_error` is the classification error of the final learned ensemble on test data

Note: You may use sklearns 'DecisonTreeClassifier' but **not** 'RandomForestClassifier' or any other bagging function



In [ ]:
def out_of_bag(trees, pointsUsed, X_train, y_train):
  error = 0
  count = 0
  for i in range(0, X_train.shape[0]):
    numTrees = 0
    predict = 0
    for t in range(0, len(trees)):
      treePoints = pointsUsed[t]
      tree = trees[t]
      if i not in treePoints:
        numTrees += 1
        x = X_train[i].reshape((1, -1))
        y_class = tree.predict(x)
        predict += y_class
    if numTrees > 0:
      predict = np.sign(predict)
      y = y_train[i]
      count += 1
      if y != predict:
        error += 1
  out_of_bag_error = error/count

  return out_of_bag_error

In [ ]:
def test(trees, X_test, y_test):
  ys = []
  for t in range(0, len(trees)):
    tree = trees[t]
    y_tree = tree.predict(X_test)
    ys.append(y_tree)
  error = 0
  for i in range(0, y_test.shape[0]):
    predict = 0
    y = y_test[i]
    for t in range(0, len(trees)):
      tree_result = ys[t]
      tree_y = tree_result[i]
      predict += tree_y
    predict = np.sign(predict)
    if predict != y:
      error += 1
  error /= y_test.shape[0]

  return error

In [ ]:
def random_forest(X_train, y_train, X_test, y_test, num_bags, m):

  # Your code here, assign the proper values to out_of_bag_error and test_error:
  trees = []
  pointsUsed = []
  for t in range(0, num_bags):
    X = np.empty(X_train.shape)
    y = np.empty(y_train.shape)
    pointIndices = []
    for n in range(0, X_train.shape[0]):
      pointIndex = random.randint(0, X_train.shape[0]-1)
      if pointIndex not in pointIndices:
        pointIndices.append(pointIndex)
      X[n] = X_train[pointIndex]
      y[n] = y_train[pointIndex]
    pointsUsed.append(pointIndices)
    tree = DecisionTreeClassifier(max_features=m)
    tree.fit(X, y)
    trees.append(tree)
  out_of_bag_error = out_of_bag(trees, pointsUsed, X_train, y_train)
  test_error = test(trees, X_test, y_test)

  return out_of_bag_error, test_error

## Run and Plot

Run your random forest code and plot figures below

In [ ]:
# Other random forest code here:
df_train = pd.read_csv("/content/gdrive/My Drive/HW5_Data_Files/zip_train.csv", header=None)
df_test = pd.read_csv("/content/gdrive/My Drive/HW5_Data_Files/zip_test.csv", header=None)
list_1_3 = [1, 3]
list_3_5 = [3, 5]

df_1_3_train = df_train[df_train[0].isin(list_1_3)]
df_1_3_train[0] = df_1_3_train[0].replace([1], -1).replace([3], 1)
df_3_5_train = df_train[df_train[0].isin(list_3_5)]
df_3_5_train[0] = df_3_5_train[0].replace([3], -1).replace([5], 1)

df_1_3_test = df_test[df_test[0].isin(list_1_3)]
df_1_3_test[0] = df_1_3_test[0].replace([1], -1).replace([3], 1)
df_3_5_test = df_test[df_test[0].isin(list_3_5)]
df_3_5_test[0] = df_3_5_test[0].replace([3], -1).replace([5], 1)

X_train_1_3 = df_1_3_train.to_numpy()
y_train_1_3 = X_train_1_3[:, 0]
X_train_1_3 = X_train_1_3[:, 1:]
X_train_3_5 = df_3_5_train.to_numpy()
y_train_3_5 = X_train_3_5[:, 0]
X_train_3_5 = X_train_3_5[:, 1:]

X_test_1_3 = df_1_3_test.to_numpy()
y_test_1_3 = X_test_1_3[:, 0]
X_test_1_3 = X_test_1_3[:, 1:]

X_test_3_5 = df_3_5_test.to_numpy()
y_test_3_5 = X_test_3_5[:, 0]
X_test_3_5 = X_test_3_5[:, 1:]

out_of_bag_error_1_3, test_error_1_3 = random_forest(X_train_1_3, y_train_1_3, X_test_1_3, y_test_1_3, 200, 100)
out_of_bag_error_3_5, test_error_3_5 = random_forest(X_train_3_5, y_train_3_5, X_test_3_5, y_test_3_5, 200, 100)

print("1, 3 OOB Error: ", out_of_bag_error_1_3)
print("1, 3 Test Error: ", test_error_1_3)
print("3, 5 OOB Error: ", out_of_bag_error_3_5)
print("3, 5 Test Error: ", test_error_3_5)

In [ ]:
bags = [1, 3, 5, 10, 20, 50, 100, 150, 200]
errors_1_3 = []
errors_3_5 = []
for b in bags:
  errors_1_3.append(random_forest(X_train_1_3, y_train_1_3, X_test_1_3, y_test_1_3, b, 100)[0])
  errors_3_5.append(random_forest(X_train_3_5, y_train_3_5, X_test_3_5, y_test_3_5, b, 100)[0])

print(errors_1_3)
print(errors_3_5)

In [ ]:
plt.plot(bags, errors_1_3)
plt.xlabel("# bags")
plt.ylabel("OOB Error")
plt.show()

In [ ]:
plt.plot(bags, errors_3_5)
plt.xlabel("# bags")
plt.ylabel("OOB Error")
plt.show()